## Local LLM Setup with Basic Setup

In [1]:
# !pip install torch
# !pip install transformers

In [2]:
# import PACKAGES
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [3]:
# You can use any transformers based model you want here
# Consider your hardware and general resource limits
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

In [4]:
# Reference Directory to use LLMs, it is up to you to assign whatever you want
# The default llm model cache directory is something 
# like that => ~/.cache/huggingface/
MODEL_REFERENCE_CACHE_DIRECTORY = "D:/JYN/EZ/EGITIM/MY_LLM_REPOSITORY/"

In [5]:
# Depending on your choices you can use cpu or gpu here
# by changing custom device map, it can be complicated
# but we use a basic "auto" mode to let the system decide what to use
CUSTOM_DEVICE_MAP = "auto"

In [6]:
# Load model
# If you connected to internet and the model doesn't exist currently in your
# model reference cache directory it will be first installed and then run
model = AutoModelForCausalLM.from_pretrained(
pretrained_model_name_or_path=MODEL_ID,
cache_dir=MODEL_REFERENCE_CACHE_DIRECTORY,
device_map=CUSTOM_DEVICE_MAP
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [7]:
# Load the tokenizer
# Tokenizer let the model unedrstand your input
# by transforming your input into something model understand, tokens
tokenizer = AutoTokenizer.from_pretrained(
pretrained_model_name_or_path=MODEL_ID,
cache_dir=MODEL_REFERENCE_CACHE_DIRECTORY
)

In [8]:
# LLM task definition
# We will generate text
TASK = "text-generation"

In [9]:
# Initialize the pipeline
# Let you merge and use model and tokenizer
pipe = pipeline(
task=TASK,
model=model,
tokenizer=tokenizer,
device_map=CUSTOM_DEVICE_MAP
)

Device set to use cpu


In [10]:
# SETUP SAMPLING PARAMETERS to use in model pipeline
# For starters these parameters are not crucial to change (maybe except max_new_tokes)
# but you can always dive deeper and understand better

# Optimized sampling parameters for faster inference (Deterministic Generation Behavior in default)
DEFAULT_DO_SAMPLING = False # Deterministic generation for speed
DEFAULT_TEMPERATURE = None # Explicitly unset temperature # Control output creativity
DEFAULT_TOP_P = None # Explicitly unset top_p # Speed up generation by limiting candidates
DEFAULT_TOP_K = None # Explicitly unset top_k
# maximum number of tokens the model will generate # can crucially affect text generation speed
DEFAULT_MAX_NEW_TOKENS = 512 
DEFAULT_REPETITION_PENALTY = 1.1 # penalizes the model for repeating the same tokens

DEFAULT_SAMPLING_PARAMS = dict(
    do_sample=DEFAULT_DO_SAMPLING,  
    temperature=DEFAULT_TEMPERATURE, 
    top_p=DEFAULT_TOP_P,  
    top_k=DEFAULT_TOP_K,
    max_new_tokens=DEFAULT_MAX_NEW_TOKENS,
    repetition_penalty=DEFAULT_REPETITION_PENALTY
)

In [11]:
# Test with only string input
# ONLY with string
llm_input_with_only_string = "Hey, tell me a short story!"

llm_output_with_only_string = pipe(
    llm_input_with_only_string,
    **DEFAULT_SAMPLING_PARAMS)

# Print output
print(llm_output_with_only_string)

[{'generated_text': "Hey, tell me a short story! I'm looking for something that's not too long and doesn't involve any superpowers or complex characters. Sure thing! Here's a short story for you:\n\nOnce upon a time, there was a young girl named Lily who lived in a small village nestled in the mountains. She had always been fascinated by nature and the world around her.\n\nOne day, while exploring the forest, she stumbled upon a mysterious cave. As she entered the cave, she felt an inexplicable pull towards it. Suddenly, she heard a voice calling out to her from within the cave.\n\nLily was terrified at first, but as she listened closer, she realized that the voice was actually a gentle breeze blowing through the air. It was carrying messages from the other side of the cave, telling her to stay inside and not venture outside.\n\nOver the next few days, Lily followed the voice's instructions and stayed hidden in the shadows of the cave. She learned about the dangers lurking in the woods

In [12]:
# As you can see the output is not directly in desired format so we will manipulate the output
llm_output_with_only_string = llm_output_with_only_string[0]['generated_text']
print(llm_output_with_only_string)

Hey, tell me a short story! I'm looking for something that's not too long and doesn't involve any superpowers or complex characters. Sure thing! Here's a short story for you:

Once upon a time, there was a young girl named Lily who lived in a small village nestled in the mountains. She had always been fascinated by nature and the world around her.

One day, while exploring the forest, she stumbled upon a mysterious cave. As she entered the cave, she felt an inexplicable pull towards it. Suddenly, she heard a voice calling out to her from within the cave.

Lily was terrified at first, but as she listened closer, she realized that the voice was actually a gentle breeze blowing through the air. It was carrying messages from the other side of the cave, telling her to stay inside and not venture outside.

Over the next few days, Lily followed the voice's instructions and stayed hidden in the shadows of the cave. She learned about the dangers lurking in the woods and the importance of being 

In [13]:
# Test with role & content conversation structure
# role content trial

llm_input_with_role_content_structure = [
    {"role":"user", 
     "content":"Hey, tell me a short story!"}]

llm_output_with_role_content_structure = pipe(
    llm_input_with_role_content_structure,
    **DEFAULT_SAMPLING_PARAMS)[0]['generated_text'] # direct output

llm_output_with_role_content_structure

[{'role': 'user', 'content': 'Hey, tell me a short story!'},
 {'role': 'assistant',
  'content': "Sure, here's a short story for you:\n\nOnce upon a time, in a small village nestled between rolling hills and dense forests, there lived a young girl named Lily. She was known throughout the village for her kindness and her love of nature.\n\nOne day, while exploring the forest, Lily stumbled upon a mysterious old tree that seemed to be holding secrets. As she approached the tree, she felt an inexplicable pull towards it, and soon found herself standing on its branches.\n\nAs she climbed higher up the tree, she discovered that the tree was not just any ordinary tree - it was home to a group of magical creatures who had been living there for centuries. These creatures were called the Nymphs, and they had come to visit Lily and their village.\n\nThe Nymphs welcomed Lily with open arms, and together they explored the forest, learning about the world around them and discovering new wonders. Th

In [14]:
# Test with role & content conversation structure
# role content trial withs system prompt

llm_input_with_role_content_structure_and_system_prompt = [
    {"role":"system", 
     "content":"You are an adventure story generator who help people to be more creative!"},
    {"role":"user", 
     "content":"Hey, tell me a short story!"}]

llm_output_with_role_content_structure_and_system_prompt = pipe(
    llm_input_with_role_content_structure_and_system_prompt,
    **DEFAULT_SAMPLING_PARAMS)[0]['generated_text'] # direct output

llm_output_with_role_content_structure_and_system_prompt

[{'role': 'system',
  'content': 'You are an adventure story generator who help people to be more creative!'},
 {'role': 'user', 'content': 'Hey, tell me a short story!'},
 {'role': 'assistant',
  'content': 'Once upon a time, in a small village nestled between rolling hills and dense forests, lived a young girl named Lily. She was known for her kind heart and love of nature. One sunny afternoon, while exploring the forest, she stumbled upon a hidden cave.\n\nInside the cave, she found a mysterious book that had been passed down through generations. As she opened it, she discovered a world of secrets and wonders beyond her wildest dreams. The book led her on a journey filled with puzzles, challenges, and unexpected adventures.\n\nAs she explored the cave, Lily encountered various creatures from the forest, including friendly pixies and mischievous trolls. Each encounter brought her closer to understanding the mysteries of the world around her.\n\nFinally, after days of wandering, Lily 